# URL Module - Malicious URL Detection (Random Forest + XGBoost)

Phase 3. Trains a Random Forest and an XGBoost
baseline on lexical URL features, compares them, and saves the stronger model
as a joblib bundle for the FastAPI inference service.

Dataset: `sid321axn/malicious-urls-dataset` (malicious_phish.csv), 651,191 URLs,
4 classes (benign, defacement, phishing, malware).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install scikit-learn xgboost joblib pandas numpy -q

In [ ]:
import re
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from xgboost import XGBClassifier

In [ ]:
# Configuration
CONFIG = {
    "csv_path":     "/content/drive/MyDrive/cm3070_datasets/malicious_phish.csv",
    "test_ratio":   0.20,
    "random_seed":  42,
    "n_estimators": 200,
    "output_path":  "/content/url_model.joblib",
}

# Class id is fixed so it matches the inference service; index = class id.
LABEL_NAMES = ["benign", "defacement", "phishing", "malware"]
LABEL_TO_ID = {name: index for index, name in enumerate(LABEL_NAMES)}

In [ ]:
# Load the dataset
print("Loading dataset...")
df = pd.read_csv(CONFIG["csv_path"])

print(f"Total rows:  {len(df):,}")
print(f"Columns:     {list(df.columns)}")
print("Class balance:")
print(df['type'].value_counts())
print(f"Nulls:       {df.isnull().sum().sum()}")

# Drop any null rows and keep only the four known classes
df = df.dropna(subset=['url', 'type'])
df = df[df['type'].isin(LABEL_NAMES)]

In [ ]:
# Feature engineering
# This block is a byte-for-byte copy of src/url_module/preprocessor.py so the
# features used here match the ones produced at inference.
FEATURE_NAMES = [
    "url_length",
    "hostname_length",
    "path_length",
    "count_dots",
    "count_hyphens",
    "count_at",
    "count_question",
    "count_equals",
    "count_percent",
    "count_slash",
    "count_digits",
    "subdomain_depth",
    "has_ip",
    "has_https",
    "has_port",
]

IP_PATTERN = re.compile(r"^(\d{1,3}\.){3}\d{1,3}$")


def extract_url_features(url):
    url = url.strip()

    # Many dataset URLs omit the scheme, so add one for parsing only.
    has_scheme = "://" in url
    scheme = url.split("://", 1)[0].lower() if has_scheme else ""
    parse_target = url if has_scheme else "http://" + url

    # Some dataset URLs are malformed such as unbalanced brackets and cannot be
    # parsed, fall back to empty host/path so the lexical counts still run.
    try:
        parsed = urlparse(parse_target)
        hostname = parsed.hostname or ""
        path = parsed.path or ""
        has_port = parsed.port is not None
    except ValueError:
        hostname = ""
        path = ""
        has_port = False

    return [
        float(len(url)),
        float(len(hostname)),
        float(len(path)),
        float(url.count(".")),
        float(url.count("-")),
        float(url.count("@")),
        float(url.count("?")),
        float(url.count("=")),
        float(url.count("%")),
        float(url.count("/")),
        float(sum(character.isdigit() for character in url)),
        float(hostname.count(".")),
        float(1 if IP_PATTERN.match(hostname) else 0),
        float(1 if scheme == "https" else 0),
        float(1 if has_port else 0),
    ]


print("Extracting features...")
feature_matrix = np.array([extract_url_features(url) for url in df['url']])
labels = df['type'].map(LABEL_TO_ID).to_numpy()
print(f"Feature matrix: {feature_matrix.shape}")

In [ ]:
# stratified split to keep classes balanced in both sets
features_train, features_test, labels_train, labels_test = train_test_split(
    feature_matrix,
    labels,
    test_size=CONFIG["test_ratio"],
    stratify=labels,
    random_state=CONFIG["random_seed"],
)

print(f"Train samples: {len(features_train):,}")
print(f"Test samples:  {len(features_test):,}")

In [ ]:
# Train the Random Forest
# Balanced weights handle the malware class, which is only ~5% of the data.
print("Training Random Forest...")
random_forest = RandomForestClassifier(
    n_estimators=CONFIG["n_estimators"],
    class_weight="balanced",
    n_jobs=-1,
    random_state=CONFIG["random_seed"],
)
random_forest.fit(features_train, labels_train)
print("Done.")

In [ ]:
print("Training XGBoost...")
xgboost_model = XGBClassifier(
    n_estimators=CONFIG["n_estimators"],
    objective="multi:softprob",
    num_class=len(LABEL_NAMES),
    tree_method="hist",
    n_jobs=-1,
    random_state=CONFIG["random_seed"],
)
xgboost_model.fit(features_train, labels_train)
print("Done.")

In [ ]:
def evaluate(model, name):
    """Print a classification report and return the macro metrics."""
    predicted = model.predict(features_test)
    print(f"\n{'='*55}")
    print(f"{name}")
    print('='*55)
    print(classification_report(
        labels_test, predicted, target_names=LABEL_NAMES
    ))
    print("Confusion matrix:")
    print(confusion_matrix(labels_test, predicted))
    return {
        "macro_f1":        f1_score(labels_test, predicted, average="macro"),
        "macro_recall":    recall_score(labels_test, predicted, average="macro"),
        "macro_precision": precision_score(labels_test, predicted, average="macro"),
    }

rf_metrics = evaluate(random_forest, "Random Forest")
xgb_metrics = evaluate(xgboost_model, "XGBoost")

# Compare against Yu et al. (2024) M-BERT: 94.42% macro-precision
print(f"\n{'Model':<16}{'macro-F1':>12}{'macro-recall':>14}{'macro-precision':>16}")
print(f"{'Random Forest':<16}{rf_metrics['macro_f1']:>12.4f}{rf_metrics['macro_recall']:>14.4f}{rf_metrics['macro_precision']:>16.4f}")
print(f"{'XGBoost':<16}{xgb_metrics['macro_f1']:>12.4f}{xgb_metrics['macro_recall']:>14.4f}{xgb_metrics['macro_precision']:>16.4f}")
print(f"{'Yu et al. 2024':<16}{'-':>12}{'-':>14}{0.9442:>16.4f}")

In [ ]:
# Save the Random Forest as the deployable model.
# Both models are evaluated above; the Random Forest is deployed because it has
# the higher macro-F1 and the higher phishing recall, and recall matters most
# in triage where a missed threat costs more than a false alarm.
print(f"Random Forest macro-F1: {rf_metrics['macro_f1']:.4f}")
print(f"XGBoost macro-F1:       {xgb_metrics['macro_f1']:.4f}")

bundle = {
    "model":         random_forest,
    "feature_names": FEATURE_NAMES,
    "label_names":   LABEL_NAMES,
    "model_version": "url-randomforest-v1",
}
joblib.dump(bundle, CONFIG["output_path"])
print(f"Saved bundle to {CONFIG['output_path']}")

In [ ]:
import os
import shutil

drive_path = "/content/drive/MyDrive/cm3070_models/url_model.joblib"
os.makedirs(os.path.dirname(drive_path), exist_ok=True)
shutil.copy(CONFIG["output_path"], drive_path)

size_mb = os.path.getsize(drive_path) / 1024 / 1024
print(f"Saved to Drive: {drive_path}  ({size_mb:.1f} MB)")
print("Download it, then place it at models/url_module/url_model.joblib")